# Teacher Labeling — GPT-4o Hop-1 Decompositions

**Goal.** Generate the training labels for the student. For each HotpotQA question I prompt
GPT-4o (the teacher) to produce the *first hop* of a reasoning decomposition as a small YAML
object: `thought`, `action`, `target_entity`. The student (Gemma-3-270M) will later learn to
imitate these.

HotpotQA ships with no reasoning traces — only questions, answers, and supporting facts. So the
teacher *synthesises* the decomposition the dataset never recorded; that synthesis is the whole
point of distillation here.

The teacher never sees the answer or the supporting facts. It works from the question alone — the
same input the student will see at inference time. I use the gold `supporting_facts` titles only
*afterwards*, to grade whether the teacher aimed at the right entity.

Everything in this notebook is plain HTTP calls to the API — no GPU needed.

## 1. Setup

The teacher is GPT-4o, so I need the OpenAI client and an API key. I read the key with `getpass`
rather than hard-coding it — the key is prompted at runtime and never written into the notebook or
its saved output, which matters since this notebook is public. It also behaves the same whether I
run on Colab, in VS Code, or locally.

In [ ]:
# !pip install -q openai pyyaml datasets
%pip install openai

In [ ]:
import os
import getpass
from openai import OpenAI

# Prompt for the key once; it lives only in memory for this session, never in the notebook.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()
print("[INFO] OpenAI client ready")

## 2. The teacher prompt

This is the instruction I give GPT-4o. It's a two-shot prompt: Example 1 is a *bridge* question
(look up the entity the question hangs on), Example 2 is a *comparison* question. For comparisons I
deliberately decompose one entity at a time — Hop-1 looks up the first entity and defers the actual
comparison to a later hop. That keeps every hop a single atomic action, so when a trajectory fails I
can localise which step broke.

The schema is fixed at three fields and `action` is always `Lookup`, which keeps the output trivial
to validate and grade.

In [ ]:
TEACHER_PROMPT = """You are an analytical reasoning agent specialized in breaking down multi-hop questions.

Your objective is to determine ONLY the first logical step (Hop 1) required to solve the question.

CRITICAL CONSTRAINTS:
1. DO NOT answer the question. Stop reasoning immediately after formulating the first hop.
2. Output your response strictly in YAML format. Do not include introductory or concluding text, markdown formatting, or conversational filler.
3. TARGET EXACT ANCHORS: The `target_entity` MUST be taken verbatim from the question. Prefer the explicitly named entity that anchors the lookup. Only when the question names no entity — when it refers to its subject by description alone — use that description verbatim. Never canonicalize, resolve, guess, or inject outside knowledge: do not expand a partial name to its full form, and do not replace a description with the real-world entity it denotes.
4. SEPARATE THE UNKNOWN: Anything you are trying to find out — whether a missing entity or a property — belongs entirely in the `thought`. The `target_entity` is only the known starting bridge the question hands you.
5. STRICT QUOTING: Every value in your YAML output (`thought`, `action`, and `target_entity`) MUST be wrapped in double quotes.

YAML SCHEMA:
thought: "[Your logical deduction of what needs to be found first about the anchor]"
action: "Lookup"
target_entity: "[The exact anchor string handed to you by the question]"

EXAMPLE 1 (Named Entity):
Question: The Oberoi family is part of a hotel company that has a head office in what city?
thought: "I need to find which hotel company the Oberoi family belongs to."
action: "Lookup"
target_entity: "Oberoi family"

EXAMPLE 2 (Comparison):
Question: Were Scott Derrickson and Ed Wood of the same nationality?
thought: "I need to find the nationality of Scott Derrickson first to eventually compare it to Ed Wood."
action: "Lookup"
target_entity: "Scott Derrickson"

EXAMPLE 3 (Anchor Move / Relational):
Question: The wife of Arthur Miller starred in what movie?
thought: "I need to find out who the wife of Arthur Miller is first."
action: "Lookup"
target_entity: "Arthur Miller"

EXAMPLE 4 (Property):
Question: Cadmium Chloride is slightly soluble in this chemical, it is also called what?
thought: "I need to find the chemical that Cadmium Chloride is slightly soluble in."
action: "Lookup"
target_entity: "Cadmium Chloride"

EXAMPLE 5 (Pure Descriptive Anchor):
Question: What language is most widely spoken in the most populous country in Africa?
thought: "I need to identify which country is the most populous in Africa first."
action: "Lookup"
target_entity: "the most populous country in Africa"
"""

print(TEACHER_PROMPT)

## 3. Pick a small set to label

I start small — 10 questions — before spending tokens on a full run. I pick a deliberate mix of 6
bridge and 4 comparison rather than just the first 10 rows, so both branches of the two-shot prompt
actually get exercised. Comparison questions are sparse near the front of the set, so I pull a larger
pool first and then sample by `type`.

In [ ]:
from datasets import load_dataset, concatenate_datasets

ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="train[:200]")

bridge_rows = ds.filter(lambda r: r["type"] == "bridge").select(range(6))
comparison_rows = ds.filter(lambda r: r["type"] == "comparison").select(range(4))
label_set = concatenate_datasets([bridge_rows, comparison_rows])

print(f"label_set: {len(label_set)} rows "
      f"({sum(r['type']=='bridge' for r in label_set)} bridge, "
      f"{sum(r['type']=='comparison' for r in label_set)} comparison)\n")
for r in label_set:
    print(f"[{r['type']:10}] {r['question']}")

## 4. Ask the teacher for a decomposition

A small function: given a question, send the teacher prompt as the system message and the question
as the user message, call GPT-4o, and return the raw YAML string it produces.

In [ ]:
def get_teacher_label(question:str) -> str:
    """Take a question string -> return the raw YAML string GPT-4o produces."""
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {"role": "system", 'content':TEACHER_PROMPT},
            {'role': 'user', 'content':f"Question: {question}"}
        ],
        temperature=0.0
    )
    return response.choices[0].message.content

In [ ]:
sample_row = label_set[6]
a = sample_row['question']

raw_gpt4o = get_teacher_label(a)

print("=== GPT-4o OUTPUT ===")
print(raw_gpt4o)

## 5. Parse and validate the output

The teacher returns a YAML string. I parse it and check it matches the schema exactly — the three
expected fields and `action == "Lookup"`. If the output is malformed I raise loudly rather than
silently dropping it, so bad traces stay visible during inspection.

In [ ]:
import yaml

EXPECTED_FIELDS = {"thought", "action", "target_entity"}

def parse_and_validate(raw: str) -> dict:
    """Parse a teacher YAML string and validate the Hop-1 schema."""
    text = raw.strip()
    # Strip a stray ```yaml fence if the model wraps the output.
    if text.startswith("```"):
        text = text.strip("`")
        text = text[text.find("\n") + 1 :] if "\n" in text else text

    try:
        obj = yaml.safe_load(text)
    except yaml.YAMLError as e:
        raise ValueError(f"Not valid YAML: {e}\n---raw---\n{raw}")

    if not isinstance(obj, dict):
        raise ValueError(f"YAML did not parse to a mapping, got {type(obj)}:\n{raw}")
    if set(obj.keys()) != EXPECTED_FIELDS:
        raise ValueError(f"Field mismatch. Expected {EXPECTED_FIELDS}, got {set(obj.keys())}")
    if obj["action"] != "Lookup":
        raise ValueError(f"action must be 'Lookup', got {obj['action']!r}")

    return obj

In [ ]:
parse_raw_gpt4o = parse_and_validate(raw_gpt4o)
print("Parsed GPT-4o output:", parse_raw_gpt4o)

## 6. Grade the target entity

This is the metric. The teacher never saw the supporting facts; here I use the gold supporting-fact
titles as an answer key and check whether the teacher's `target_entity` actually points at one of
them. How strict that match should be is the interesting design decision.

In [ ]:
JUDGE_PROMPT = """You are grading the FIRST step of a multi-hop question decomposition.

You are given:
- the QUESTION,
- a PROPOSED FIRST LOOKUP (the entity a model chose to look up first),
- the GOLD TITLES: the titles of the reference articles that hold the supporting facts for the question.

Decide one thing: would looking up the PROPOSED FIRST LOOKUP be a correct first step, one that leads to the gold supporting facts?

Count it as correct when the proposed lookup is the right starting entity, including when:
- it is the same real-world entity as a gold title under a different name,
- it is a description that denotes a gold entity,
- it is the right anchor even though the relevant fact is filed under a related gold title, 
- the question compares two named entities and the lookup is either one of them, since looking up either is a valid first step and which one
turns out to be the answer does not matter, or
- the question names no entity and the lookup is a faithful description of its subject (resolving that description is a later step, not part of this decision).

Count it as incorrect when the proposed lookup is only loosely related, is the unknown the question is trying to find, or is the wrong entity.

Reason briefly first, then give your verdict.
Return ONLY a JSON object: {"reason": "<one sentence>", "match": <true or false>}"""

In [ ]:
import json

def grade_target_entity_llm(question: str, target_entity: str, gold_titles: list) -> dict:
    """LLM-judge: is target_entity a correct Hop-1 lookup for this question,
    given the gold supporting-fact titles? Return {'reason': str, 'match': bool}."""
    user = (
        f"Question: {question}\n"
        f"PROPOSED FIRST LOOKUP: {target_entity}\n"
        f"GOLD TITLES: {gold_titles}"
    )
    resp = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": user},
        ],
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

## 7. Run it and inspect

Wire the pieces together over the 10-row set and read every trace by hand. What I see here decides
the open questions: whether to keep `easy` rows, and how the teacher behaves when the Hop-1 entity
is described rather than named.

In [ ]:
hits = 0
for row in label_set:
    question = row['question']
    gold_titles = row['supporting_facts']['title'] # the title LiST, not the dict
    try:
        raw = get_teacher_label(question)
        parsed = parse_and_validate(raw)
        match = grade_target_entity_llm(question, parsed['target_entity'], gold_titles)
    except Exception as e:
        print(f"[{row['type']:10}] {question}\n   !! FAILED: {e}\n")
        continue

    hits += match['match']
    print(f"[{row['type']:10}] {question}")
    print(f"   target_entity :  {parsed['target_entity']!r}")
    print(f"   gold titles   :  {sorted(set(gold_titles))}")
    print(f"   match         :  {match['match']}")
    print(f"   reason        :  {match['reason']}\n")


print(f"LLM_judge hits: {hits}/{len(label_set)}")

## 8. Scale the run

The 10-row loop above was for reading traces by hand. Now I run the same teacher → parse → judge
pipeline over a larger sample and write the labels to disk. Three choices shape this run:

- **500 questions**, at HotpotQA's natural mix of roughly 80% bridge and 20% comparison, so the
  student sees the same balance it will meet at inference.
- **Keep only the rows the judge passes.** The student trains on clean Hop-1 labels; every rejected
  row is written to a separate file with its reason so I can read why it failed.
- **Append as it goes, with resume.** Each result is written the moment it is graded, and a re-run
  skips ids already on disk, so a dropped connection partway through does not cost the whole run.

The output is `data/hop1_labeled.jsonl` (the training set) and `data/hop1_rejected.jsonl` (the audit
trail). The reject file is where I confirm two things I have not yet seen at scale: whether a colon
inside a `thought` ever breaks the YAML parse, and whether the judge produces any false negatives.

In [ ]:
from pathlib import Path

N_TOTAL = 500
BRIDGE_FRAC = 0.8
n_bridge = int(N_TOTAL * BRIDGE_FRAC)        # 400
n_comparison = N_TOTAL - n_bridge            # 100

# Comparison rows are sparse, so pull a wide pool first and sample each type by count.
pool = load_dataset("hotpotqa/hotpot_qa", "distractor", split="train[:1200]")
bridge_pool = pool.filter(lambda r: r["type"] == "bridge")
comparison_pool = pool.filter(lambda r: r["type"] == "comparison")

assert len(bridge_pool) >= n_bridge and len(comparison_pool) >= n_comparison, \
    "pool slice too small for the requested counts; widen the train[:N] slice"

scaled_set = concatenate_datasets([
    bridge_pool.select(range(n_bridge)),
    comparison_pool.select(range(n_comparison)),
])
print(f"scaled_set: {len(scaled_set)} rows "
      f"({sum(r['type']=='bridge' for r in scaled_set)} bridge, "
      f"{sum(r['type']=='comparison' for r in scaled_set)} comparison)")

In [ ]:
import json
from tqdm.auto import tqdm

OUT_DIR = Path("../data")
OUT_DIR.mkdir(exist_ok=True)
ACCEPTED = OUT_DIR / "hop1_labeled.jsonl"
REJECTED = OUT_DIR / "hop1_rejected.jsonl"

def done_ids(*paths) -> set:
    """ids already written on a previous run, so a re-run resumes instead of repeating work."""
    seen = set()
    for p in paths:
        if p.exists():
            for line in p.read_text().splitlines():
                if line.strip():
                    seen.add(json.loads(line)["id"])
    return seen

def append(path, record):
    with path.open("a") as f:
        f.write(json.dumps(record) + "\n")

already = done_ids(ACCEPTED, REJECTED)
accepted = rejected = 0

for row in tqdm(scaled_set, desc="labeling"):
    if row["id"] in already:
        continue
    base = {
        "id": row["id"],
        "question": row["question"],
        "type": row["type"],
        "level": row["level"],
        "gold_titles": sorted(set(row["supporting_facts"]["title"])),
    }

    try:
        raw = get_teacher_label(row["question"])
    except Exception as e:
        append(REJECTED, {**base, "stage": "teacher_api", "reason": str(e)})
        rejected += 1
        continue

    try:
        parsed = parse_and_validate(raw)
    except Exception as e:
        append(REJECTED, {**base, "stage": "parse", "reason": str(e), "raw": raw})
        rejected += 1
        continue

    try:
        verdict = grade_target_entity_llm(row["question"], parsed["target_entity"], row["supporting_facts"]["title"])
    except Exception as e:
        append(REJECTED, {**base, "stage": "judge_api", "reason": str(e), "target_entity": parsed["target_entity"]})
        rejected += 1
        continue

    if verdict["match"]:
        append(ACCEPTED, {**base, **parsed, "judge_reason": verdict["reason"]})
        accepted += 1
    else:
        append(REJECTED, {**base, "stage": "judge", "target_entity": parsed["target_entity"], "reason": verdict["reason"]})
        rejected += 1

print(f"accepted: {accepted}  rejected: {rejected}  on disk now: {len(done_ids(ACCEPTED, REJECTED))}")

In [ ]:
import pandas as pd

labeled = pd.read_json(ACCEPTED, lines=True)
print(f"{len(labeled)} accepted labels")
print(labeled["type"].value_counts().to_string())
display(labeled[["question", "target_entity", "thought"]].head(10))

# Read the rejects by hand: parse-stage rows test the strict-quoting fix, judge-stage rows
# are the ones to eyeball for false negatives at scale.
if REJECTED.exists():
    rejects = pd.read_json(REJECTED, lines=True)
    print(f"\n{len(rejects)} rejected")
    print(rejects["stage"].value_counts().to_string())
    display(rejects[["question", "stage", "reason"]].head(20))

In [ ]:
print(rejects[["question", "stage", "reason"]].head(20).to_string())

## 9. Format into chat messages and split

The labels are flat JSON. `SFTTrainer` trains on *conversations*: each example is a list of turns,
each turn a `{role, content}` dict, and that shape is what the Gemma chat template renders into the
tokens the model actually learns from. So formatting means turning every labeled row into a
`messages` list — the question becomes the `user` turn, the YAML decomposition becomes the
`assistant` turn.

The tutorial I follow puts the target under a `system` role; I use `assistant` instead, because the
target is the model's own reply and Gemma builds the model turn from the `assistant` role.

I build two versions of the dataset so I can settle one open question by experiment instead of guess:
does a short fixed system instruction make the student's output more stable, or is the question
alone enough? One version prepends a `system` turn, the other does not. I train both, compare
structured-output adherence on the held-out test set, and keep the winner — and if they tie I keep
the no-system version, since it is simpler and needs no instruction kept in sync at inference.

The assistant target reproduces the teacher's YAML verbatim, double quotes and all, so what the
student learns to emit is exactly what the judge already graded.

In [ ]:
from datasets import load_dataset

In [ ]:
# The assistant target is the teacher's YAML, rebuilt verbatim. The labels carry no embedded
# quotes or newlines (checked), so plain double-quoting reproduces the strict-quoted format safely.
def to_yaml_target(row):
    return (
        f'thought: "{row["thought"]}"\n'
        f'action: "{row["action"]}"\n'
        f'target_entity: "{row["target_entity"]}"'
    )

# Short, fixed instruction — used only by the with-system variant of the ablation.
SYSTEM_MSG = (
    "You are a reasoning agent. Given a multi-hop question, output only the "
    "first reasoning step as YAML with three fields: thought, action, target_entity."
)

def to_messages(row, include_system):
    """Flat label row -> {'messages': [...]}. include_system toggles the two ablation variants."""
    msgs = []
    if include_system:
        msgs.append({"role": "system", "content": SYSTEM_MSG})
    msgs.append({"role": "user", "content": row["question"]})
    msgs.append({"role": "assistant", "content": to_yaml_target(row)})
    return {"messages": msgs}

labeled_ds = load_dataset("json", data_files="../data/hop1_labeled.jsonl", split="train")
ds_nosys = labeled_ds.map(lambda r: to_messages(r, include_system=False))
ds_sys   = labeled_ds.map(lambda r: to_messages(r, include_system=True))

# Eyeball one example of each before training on it.
print("NO-SYSTEM:")
for turn in ds_nosys[0]["messages"]:
    print(f"  [{turn['role']}] {turn['content']!r}")
print("\n*"*3)
print("\nWITH-SYSTEM:")
for turn in ds_sys[0]["messages"]:
    print(f"  [{turn['role']}] {turn['content']!r}")

In [ ]:
# 80/20 split. Default shuffle is ON; I keep it (not the tutorial's shuffle=False) because the
# labels are ordered bridge-first then comparison, so an unshuffled tail-split would put almost
# all comparison rows into the test set. The fixed seed gives both variants the same membership,
# so the only difference between the two trained models is the system turn.
split_nosys = ds_nosys.train_test_split(test_size=0.2, seed=42)
split_sys   = ds_sys.train_test_split(test_size=0.2, seed=42)

# Confirm both types land in train and test before saving anything.
import pandas as pd
for part in ["train", "test"]:
    mix = pd.Series(split_nosys[part]["type"]).value_counts(normalize=True).round(2).to_dict()
    print(f"{part:5} {len(split_nosys[part]):3} rows   type mix {mix}")

# Save all four splits for the Colab training notebook. messages survives the JSONL round-trip
# as nested JSON, so SFTTrainer still reads a conversational dataset.
for name, split in [("nosys", split_nosys), ("sys", split_sys)]:
    split["train"].to_json(f"../data/train_{name}.jsonl")
    split["test"].to_json(f"../data/test_{name}.jsonl")
print("\nwrote train_/test_ x nosys/sys (.jsonl) to ../data/")